# CodeList Evaluation Module

To evaluate the quality of the generated codelists, in absence of clinicians to review our outputs, we decided to search for published papers that document the CodeList used to interrogate databases with their research questions. The question would be used as input query in our workflow, and the published codelist assumed to be the golden standard outcome, that our system should be able to retrieve given similar search terms.

This module compares codelists obtained at different stages of the workflow against the published reference codelist.

Confusion matrixes are plotted at each interrogation. Recall should be prioritized given the importance of getting an exhaustive list, and worse outcome of missing any relevant codes out.

# Packages

In [1]:
import pandas as pd
import requests
from google.colab import files
import json

pd.set_option('display.max_colwidth', None)

# Reference Codelist Import

## Import

This function imports any individual json file created from Evaluation TestSet Reformatting and saves it as a dataframe. It also extracts the Research Question, to pass as User Query

In [2]:
def load_json_and_extract_query():
    # Upload file
    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]

    # Load JSON
    with open(file_name, 'r') as f:
        data = json.load(f)

    # Convert to DataFrame
    df = pd.DataFrame(data)

    # Extract Research_question (assumes same value across rows)
    if "Research_question" in df.columns:
        user_query = df["Research_question"].dropna().iloc[0]
    else:
        user_query = None

    return df, user_query

In [3]:
# Run the function
df_loaded, user_query = load_json_and_extract_query()

print(user_query)
df_loaded.head()

Saving Source_entry_9.json to Source_entry_9.json
ICD10 codes for a clinical diagnosis of delirium;


,Entry_no,Ref_link,Ref_reviewer,Ref_review_FLAG,Source_creation_date,Research_question,Codelist_vocabulary,Codelist,Codelist_terms,Terms_count,Codes_count,Diff_FLAG
0,9,https://datacompass.lshtm.ac.uk/id/eprint/4529/,AD,no - mock,1569715200000,ICD10 codes for a clinical diagnosis of delirium;,ICD-10,R26,Delirium,1,1,N


## Deduplication Step


In [4]:
# Deduplicate Clinical_codes, concatenating Characteristics with "/" as a divider when multiple rows are merged
df_ref = df_loaded.groupby('Codelist')['Codelist_terms'].apply(lambda x: '/ '.join(x)).reset_index()
df_ref

,Codelist,Codelist_terms
0,R26,Delirium


# Run Pipeline via API

Call the live NICE Clinical Code List Generator at `clinicalcodes.uk`. The `/api/evaluate` endpoint accepts the uploaded test set, runs the full RAG pipeline on the `Research_question`, and returns recall/precision/F1 against the gold-standard codelist.

In [ ]:
API_BASE = "https://clinicalcodes.uk/api"

# Send the uploaded test set to the evaluate endpoint
with open(file_name, 'r') as f:
    test_set_data = json.load(f)

print(f"Sending query: {user_query}")
print(f"Reference codes: {len(test_set_data)} entries")
print("Running pipeline (this may take 30-60 seconds)...")

response = requests.post(
    f"{API_BASE}/evaluate",
    json={"test_set": test_set_data},
    timeout=300,
)
response.raise_for_status()
eval_result = response.json()

print(f"
Done in {eval_result.get('elapsed_seconds', '?')}s")
print(f"
{eval_result['summary']}")

# Evaluation Results

In [ ]:
# Display metrics for each evaluation stage
stages = eval_result.get("stages", {})
rows = []
for stage_name, metrics in stages.items():
    rows.append({
        "Stage": stage_name,
        "Ref Codes": metrics["n_ref_codes"],
        "Output Codes": metrics["n_output_codes"],
        "Recall": f"{metrics['recall']:.1%}",
        "Precision": f"{metrics['precision']:.1%}",
        "F1": f"{metrics['f1']:.1%}",
        "TP": metrics["tp_count"],
        "FP": metrics["fp_count"],
        "FN": metrics["fn_count"],
    })

df_api_metrics = pd.DataFrame(rows)
print("=== Evaluation Metrics (via API) ===")
df_api_metrics

In [ ]:
# False exclusions: reference codes the LLM incorrectly excluded
false_exc = eval_result.get("false_exclusions", {})
print(f"False exclusions: {false_exc['count']} reference codes were incorrectly excluded")
if false_exc.get("codes"):
    df_false_exc = pd.DataFrame(false_exc["codes"])
    display(df_false_exc)

# Uncertain codes that were in the reference
unc_ref = eval_result.get("uncertain_in_reference", {})
print(f"
Uncertain in reference: {unc_ref['count']} codes marked uncertain were in the gold standard")
if unc_ref.get("codes"):
    df_unc_ref = pd.DataFrame(unc_ref["codes"])
    display(df_unc_ref)

In [ ]:
# TP / FP / FN detail for the included_only stage
inc_stage = stages.get("included_only", {})

if inc_stage.get("tp_codes"):
    print("=== True Positives (correctly retrieved) ===")
    display(pd.DataFrame(inc_stage["tp_codes"]))

if inc_stage.get("fn_codes"):
    print("
=== False Negatives (missed reference codes) ===")
    display(pd.DataFrame(inc_stage["fn_codes"]))

if inc_stage.get("fp_codes"):
    print(f"
=== False Positives ({len(inc_stage['fp_codes'])} extra codes retrieved) ===")
    display(pd.DataFrame(inc_stage["fp_codes"]).head(20))